# Problem 4 — Cryptanalysis of Many-Time Pad

Source: `project2-MTP-problems-v2026B.docx(1).pdf`, pages 2–4. The ten numbered ciphertexts are in `data/ciphertexts.txt`; the separately labeled target is in `data/target.txt`.

**Status:** official inputs imported; iterative attack tools ready; no plaintext cribs accepted yet.

The PDF says “last message” but prints a distinct target after ciphertext 10. This notebook treats the explicitly labeled target as TARGET and retains C10 unchanged.

Use Python 3.9 or newer. Run from `Many_Time_Pad/` or the repository root. The code reads the official dataset and does not overwrite it.


## Strategy and assumptions

For reused pad bytes at the same offsets, `C_i XOR C_j = M_i XOR M_j`. Unequal message lengths are handled by comparing their overlapping prefix.

1. Inspect lengths and pairwise XORs.
2. Rank possible spaces using the letter/space case-toggle pattern in Hint H.
3. Drag a guessed English segment (a crib) across a chosen XOR pair, as required by Hint J.
4. Test a guessed plaintext segment against every overlapping ciphertext through the implied key bytes.
5. Accept supported cribs explicitly, rerun, and inspect partial target recovery.
6. Record unsuccessful guesses, contradictions, adjustments, and remaining unknowns in `analysis_notes.md`.

A high space score or readable fragment is a clue, not proof. The character filter uses ASCII letters, digits (Hint I), space, comma, period, and question mark. This is a candidate filter, not a model of English grammar. If the professor clarifies that digits are excluded, remove them from `ALLOWED`.


In [1]:
from pathlib import Path
from itertools import combinations
from collections import Counter
from string import ascii_letters, digits
import json
import hashlib

ALLOWED = set((ascii_letters + digits + " ,.?").encode("ascii"))


def load_hex_lines(path):
    values = []
    for line_number, line in enumerate(path.read_text(encoding="ascii").splitlines(), 1):
        text = "".join(line.split())
        if not text:
            continue
        try:
            values.append(bytes.fromhex(text))
        except ValueError as exc:
            raise ValueError(f"{path.name}, line {line_number}: invalid hexadecimal.") from exc
    if not values:
        raise ValueError(f"{path.name}: no ciphertexts found.")
    return values


def xor_overlap(left, right):
    return bytes(a ^ b for a, b in zip(left, right))


def preview(data):
    return "".join(chr(value) if 32 <= value <= 126 else "·" for value in data)


def is_ascii_letter(value):
    return 65 <= value <= 90 or 97 <= value <= 122


In [2]:
workspace = next(
    (folder for folder in [Path("Many_Time_Pad"), Path(".")]
     if (folder / "data" / "ciphertexts.txt").is_file()),
    None,
)
if workspace is None:
    raise FileNotFoundError("Run from the repository root or Many_Time_Pad/.")

numbered = load_hex_lines(workspace / "data" / "ciphertexts.txt")
targets = load_hex_lines(workspace / "data" / "target.txt")
if len(numbered) != 10 or len(targets) != 1:
    raise ValueError("Expected ten numbered ciphertexts and one separately labeled target.")

messages = {f"C{index:02d}": value for index, value in enumerate(numbered, 1)}
messages["TARGET"] = targets[0]
target = messages["TARGET"]

provenance = json.loads((workspace / "data" / "provenance.json").read_text())
if {record["label"] for record in provenance["records"]} != set(messages):
    raise ValueError("Provenance labels do not match the dataset.")
for record in provenance["records"]:
    value = messages[record["label"]]
    if value.hex() != record["hex"] or hashlib.sha256(value).hexdigest() != record["sha256"]:
        raise ValueError(f"Source integrity mismatch: {record['label']}")

print("Label | Length in bytes")
for label, value in messages.items():
    print(f"{label:6s} | {len(value)}")
print("Source integrity checks passed.")


Label | Length in bytes
C01    | 78
C02    | 113
C03    | 90
C04    | 90
C05    | 103
C06    | 98
C07    | 110
C08    | 46
C09    | 123
C10    | 52
TARGET | 59
Source integrity checks passed.


## Inspect one XOR pair

Change these labels to explore another pair, including TARGET. Offsets are zero-based byte positions. Nonprintable output is shown as a middle dot; it is not recovered text.


In [3]:
LEFT_LABEL = "C01"
RIGHT_LABEL = "TARGET"

if LEFT_LABEL == RIGHT_LABEL:
    raise ValueError("Choose different messages.")
pair = xor_overlap(messages[LEFT_LABEL], messages[RIGHT_LABEL])
print(f"{LEFT_LABEL} XOR {RIGHT_LABEL}, {len(pair)} overlapping bytes")
print("Hex:", pair.hex())
print("Preview:", preview(pair))
print("Offset | XOR byte | Preview")
for offset, value in enumerate(pair):
    print(f"{offset:6d} | {value:02x}       | {preview(bytes([value]))}")

print("\nPair | Overlap | Hex preview")
for (left_label, left), (right_label, right) in combinations(messages.items(), 2):
    result = xor_overlap(left, right)
    print(f"{left_label}/{right_label} | {len(result)} | {result[:12].hex()}")


C01 XOR TARGET, 59 overlapping bytes
Hex: 00001c4e571400491a0f45414e1603522c354d4f050f190b0c134c4e0d19160249011209490e0d011f4b2d1c000c0c110c441106150f4c03174311
Preview: ···NW··I··EAN··R,5MO······LN····I···I····K-······D····L··C·
Offset | XOR byte | Preview
     0 | 00       | ·
     1 | 00       | ·
     2 | 1c       | ·
     3 | 4e       | N
     4 | 57       | W
     5 | 14       | ·
     6 | 00       | ·
     7 | 49       | I
     8 | 1a       | ·
     9 | 0f       | ·
    10 | 45       | E
    11 | 41       | A
    12 | 4e       | N
    13 | 16       | ·
    14 | 03       | ·
    15 | 52       | R
    16 | 2c       | ,
    17 | 35       | 5
    18 | 4d       | M
    19 | 4f       | O
    20 | 05       | ·
    21 | 0f       | ·
    22 | 19       | ·
    23 | 0b       | ·
    24 | 0c       | ·
    25 | 13       | ·
    26 | 4c       | L
    27 | 4e       | N
    28 | 0d       | ·
    29 | 19       | ·
    30 | 16       | ·
    31 | 02       | ·
    32 | 49       | I
    33 | 01     

## Rank possible spaces

If one plaintext has a space and the other has a letter at an offset, their XOR is a letter with its case toggled. Count such observations across the other messages. Punctuation, digits, and limited overlap can weaken the heuristic.

The cell lists candidates only. It does not accept key bytes or decrypt the target automatically.


In [4]:
def space_candidates(messages, min_comparisons=6, threshold=0.75):
    rows = []
    for label, ciphertext in messages.items():
        for offset, byte in enumerate(ciphertext):
            peers = [other[offset] for other_label, other in messages.items()
                     if other_label != label and offset < len(other)]
            votes = sum(is_ascii_letter(byte ^ peer) for peer in peers)
            if len(peers) < min_comparisons or votes / len(peers) < threshold:
                continue
            candidate_key = byte ^ 0x20
            violations = sum((peer ^ candidate_key) not in ALLOWED for peer in peers)
            rows.append({
                "label": label, "offset": offset, "votes": votes,
                "comparisons": len(peers), "ratio": votes / len(peers),
                "key_byte": candidate_key, "violations": violations,
            })
    return sorted(rows, key=lambda row: (-row["ratio"], -row["comparisons"],
                                         row["offset"], row["label"]))


MIN_COMPARISONS = 6
SPACE_THRESHOLD = 0.75
space_rows = space_candidates(messages, MIN_COMPARISONS, SPACE_THRESHOLD)
print("Label | Offset | Votes/Peers | Implied key | Character-filter violations")
for row in space_rows[:30]:
    print(f"{row['label']} | {row['offset']} | {row['votes']}/{row['comparisons']} | "
          f"{row['key_byte']:02x} | {row['violations']}")
print(f"{len(space_rows)} candidates at the current threshold; none accepted.")


Label | Offset | Votes/Peers | Implied key | Character-filter violations
C04 | 2 | 10/10 | 63 | 0
C03 | 5 | 10/10 | e8 | 0
C01 | 15 | 10/10 | a2 | 0
TARGET | 19 | 10/10 | a0 | 0
C07 | 20 | 10/10 | 95 | 0
TARGET | 26 | 10/10 | f8 | 0
C06 | 29 | 10/10 | 6a | 0
TARGET | 32 | 10/10 | ec | 0
C09 | 33 | 10/10 | ac | 0
C03 | 35 | 10/10 | 96 | 0
C06 | 37 | 10/10 | 8b | 0
C02 | 38 | 10/10 | 78 | 0
C05 | 56 | 8/8 | ca | 0
C04 | 59 | 7/7 | 3e | 0
C07 | 61 | 7/7 | c8 | 0
C05 | 67 | 7/7 | 81 | 0
C01 | 69 | 7/7 | b8 | 0
C06 | 72 | 7/7 | 38 | 0
C04 | 75 | 7/7 | 27 | 0
C04 | 78 | 6/6 | fa | 0
C06 | 79 | 6/6 | 60 | 0
C06 | 81 | 6/6 | 28 | 0
C07 | 83 | 6/6 | 92 | 0
C03 | 84 | 6/6 | 61 | 0
C04 | 85 | 6/6 | af | 0
C07 | 86 | 6/6 | 54 | 0
C03 | 88 | 6/6 | aa | 0
C04 | 6 | 9/10 | 7b | 0
C09 | 6 | 9/10 | 7b | 0
C05 | 8 | 9/10 | 12 | 0
93 candidates at the current threshold; none accepted.


## Crib dragging

Enter a guessed word or phrase in CRIB. For each placement in the selected pair, XOR it with the pairwise XOR to obtain a candidate fragment of the other plaintext. The filter keeps fragments within ALLOWED.

Each result has two possible orientations: the crib may belong to either message. A readable result still needs cross-checking. An empty CRIB leaves the notebook in preparation mode.


In [5]:
def crib_drag(left, right, crib):
    encoded = crib.encode("ascii")
    if not encoded or any(value not in ALLOWED for value in encoded):
        raise ValueError("Provide a nonempty crib using the configured characters.")
    pair = xor_overlap(left, right)
    rows = []
    for offset in range(len(pair) - len(encoded) + 1):
        other = xor_overlap(pair[offset:offset + len(encoded)], encoded)
        if all(value in ALLOWED for value in other):
            rows.append((offset, other.decode("ascii")))
    return rows


CRIB = ""
if CRIB:
    for offset, other in crib_drag(messages[LEFT_LABEL], messages[RIGHT_LABEL], CRIB):
        print(f"Offset {offset}: crib={CRIB!r}, other candidate={other!r}")
else:
    print("Set CRIB to an English guess, then rerun this cell.")


Set CRIB to an English guess, then rerun this cell.


## Cross-check one plaintext hypothesis

Set GUESS_LABEL, GUESS_OFFSET, and GUESS_TEXT. Derive the implied key fragment, then inspect what it produces at those offsets in all other messages.

Character-filter failures are contradictions under the configured alphabet. Zero failures alone do not establish an English sentence or a correct guess.


In [6]:
def evaluate_crib(messages, label, offset, text):
    encoded = text.encode("ascii")
    if label not in messages:
        raise ValueError("Unknown message label.")
    if not encoded or any(value not in ALLOWED for value in encoded):
        raise ValueError("Provide a nonempty crib using the configured characters.")
    if offset < 0 or offset + len(encoded) > len(messages[label]):
        raise ValueError("The crib must fit entirely inside the selected message.")
    key_fragment = xor_overlap(messages[label][offset:offset + len(encoded)], encoded)
    rows = []
    for other_label, ciphertext in messages.items():
        overlap = ciphertext[offset:offset + len(encoded)]
        if not overlap:
            continue
        plaintext = xor_overlap(overlap, key_fragment)
        rows.append({
            "label": other_label, "fragment": preview(plaintext),
            "violations": sum(value not in ALLOWED for value in plaintext),
            "bytes_checked": len(plaintext),
        })
    return key_fragment, rows


GUESS_LABEL = "TARGET"
GUESS_OFFSET = 0
GUESS_TEXT = ""
if GUESS_TEXT:
    key_fragment, checks = evaluate_crib(messages, GUESS_LABEL, GUESS_OFFSET, GUESS_TEXT)
    print("Implied key fragment:", key_fragment.hex())
    for row in checks:
        print(row)
else:
    print("Set GUESS_TEXT after inspecting crib or space candidates.")


Set GUESS_TEXT after inspecting crib or space candidates.


## Explicitly accepted cribs and partial recovery

Add supported guesses to ACCEPTED_CRIBS as dictionaries with label, offset, text, and evidence. Rebuild the candidate key from scratch on each run so rejected guesses do not linger.

Conflicting key bytes or character-filter contradictions raise an error. These checks cannot prove that a plausible guess is correct. Question marks mark unknown byte positions in the partial display; consult the count and offsets to distinguish them from an actual recovered question mark.


In [7]:
def build_candidate_key(messages, accepted_cribs):
    key = [None] * max(map(len, messages.values()))
    for crib in accepted_cribs:
        if not crib.get("evidence", "").strip():
            raise ValueError("Record evidence for each accepted crib.")
        fragment, checks = evaluate_crib(
            messages, crib["label"], crib["offset"], crib["text"]
        )
        if any(row["violations"] for row in checks):
            raise ValueError("Crib contradicts the configured character filter.")
        for offset, value in enumerate(fragment, crib["offset"]):
            if key[offset] is not None and key[offset] != value:
                raise ValueError(f"Conflicting key hypotheses at byte offset {offset}.")
            key[offset] = value
    return key


def partial_decrypt(ciphertext, candidate_key):
    return "".join(
        "?" if candidate_key[offset] is None
        else chr(byte ^ candidate_key[offset])
        for offset, byte in enumerate(ciphertext)
    )


ACCEPTED_CRIBS = []
candidate_key = build_candidate_key(messages, ACCEPTED_CRIBS)
unknown_offsets = [offset for offset in range(len(target))
                   if candidate_key[offset] is None]
print("Target with unknown positions marked ?")
print(partial_decrypt(target, candidate_key))
print(f"Candidate key coverage for target: {len(target) - len(unknown_offsets)}/{len(target)} bytes")
print("Unknown offsets:", unknown_offsets)


Target with unknown positions marked ?
???????????????????????????????????????????????????????????
Candidate key coverage for target: 0/59 bytes
Unknown offsets: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58]


## Outcome log and report

Update `analysis_notes.md` after each meaningful iteration. Explain what worked, what failed, the effect of changing thresholds or cribs, and unresolved positions. Do not describe candidate output as verified recovery without evidence.

The project requires a written explanation of all four problems, notebooks, and PDF submission. Problem 4 has 5 of the 10 rubric points, including 2 points for correctly decrypting the target. This starter does not satisfy that recovery requirement yet.

All team members must review and sign off on all problems. See `assignment_requirements.md` for the complete preparation checklist.
